<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0;">Lab 03: Add Local Function Tools to Your MAF Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">Give the travel agent in-process access to flight, hotel, and car rental data using local Python tools</p>
</div>

**What We Learn:** How to define Python functions as local tools that execute inside the agent process — giving the Contoso Travel agent fast, in-process access to flight, hotel, and car rental data without external round-trips.

---

## 🧳 The Contoso Travel Story

The Contoso Travel concierge can now hold conversations, but it can't actually look up real inventory. In the prompted agents track, we used FunctionTool definitions that Foundry called externally. With MAF, tools run inside the agent process itself — no network round-trips, full access to local data, and the ability to run complex Python logic.

- **The Problem:** The agent needs to search real flight, hotel, and car rental data. With prompted agents, every tool call was an external round-trip. The team wants tools that execute locally for lower latency and more flexibility.
- **This Lab Solves:** Defining Python functions as local tools using `Annotated` type hints, wiring them into the MAF `Agent` class, and testing multi-tool queries against Contoso Travel inventory.

## 1. Setup

Connect to Microsoft Foundry and configure the project client.

---

In [ ]:
# Connect to Microsoft Foundry and get OpenAI client
import os
import json
import pandas as pd
from typing import Annotated
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1-mini")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print(f"✅ Connected to Foundry project")

## 2. Load Contoso Travel Data

Load the flight, hotel, and car rental CSV files that the tool functions will query.

---

In [ ]:
# Load inventory data — same CSVs used in prompted agents labs
data_path = "../../data/contoso-travel"
flights_df = pd.read_csv(f"{data_path}/flights.csv")
hotels_df = pd.read_csv(f"{data_path}/hotels.csv")
cars_df = pd.read_csv(f"{data_path}/car_rentals.csv")

print(f"📊 Loaded: {len(flights_df)} flights, {len(hotels_df)} hotels, {len(cars_df)} car rentals")

## 3. Define Tool Functions

MAF uses <code>Annotated</code> type hints to define tool parameters — the framework auto-generates JSON schemas from these annotations. Each function becomes a local tool that executes inside the agent process.

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>💡 Tip:</b> The <code>Annotated[str, "description"]</code> syntax serves double duty — Python gets the type, and MAF gets the parameter description for the generated JSON schema.
</div>

### The Tool-Call Loop

When you pass tool functions to the MAF <code>Agent</code>, it manages a loop automatically:

```
┌──────────────┐     ┌───────────────────────┐     ┌──────────────────┐
│  User query  │ ──▶ │  Model (gpt-4.1-mini) │ ──▶ │  Detect tool     │
│              │     │  analyzes the query    │     │  call request    │
└──────────────┘     └───────────────────────┘     └────────┬─────────┘
                                                            │
                     ┌───────────────────────┐              │
                     │  Model generates      │     ┌────────▼─────────┐
                     │  final answer using    │ ◀── │  Execute Python  │
                     │  tool results          │     │  function locally│
                     └───────────────────────┘     │  (return JSON)   │
                                                   └──────────────────┘
```

The model may call multiple tools in a single turn if the query requires it.

---

In [ ]:
# Define tool functions with Annotated type hints for auto schema generation
def search_flights(
    origin: Annotated[str, "Departure city name"] = "",
    destination: Annotated[str, "Arrival city name"] = "",
    max_price: Annotated[float, "Maximum price in USD"] = 10000,
) -> str:
    """Search Contoso Travel flight inventory by origin, destination, and price."""
    results = flights_df.copy()
    if origin:
        results = results[results["origin"].str.lower().str.contains(origin.lower())]
    if destination:
        results = results[results["destination"].str.lower().str.contains(destination.lower())]
    results = results[results["price_usd"] <= max_price]

    if results.empty:
        return json.dumps({"message": "No flights found matching your criteria."})
    return results.head(5).to_json(orient="records", indent=2)


def search_hotels(
    city: Annotated[str, "City name to search"] = "",
    min_stars: Annotated[int, "Minimum star rating (1-5)"] = 1,
    max_price: Annotated[float, "Maximum price per night in USD"] = 10000,
) -> str:
    """Search Contoso Travel hotel inventory by city, star rating, and price."""
    results = hotels_df.copy()
    if city:
        results = results[results["city"].str.lower().str.contains(city.lower())]
    results = results[results["star_rating"] >= min_stars]
    results = results[results["price_per_night_usd"] <= max_price]

    if results.empty:
        return json.dumps({"message": "No hotels found matching your criteria."})
    return results.head(5).to_json(orient="records", indent=2)


def search_car_rentals(
    city: Annotated[str, "Pickup city name"] = "",
    car_type: Annotated[str, "Vehicle type: Economy, SUV, Luxury, or Minivan"] = "",
) -> str:
    """Search Contoso Travel car rental inventory by city and vehicle type."""
    results = cars_df.copy()
    if city:
        results = results[results["city"].str.lower().str.contains(city.lower())]
    if car_type:
        results = results[results["car_type"].str.lower() == car_type.lower()]
    results = results[results["available"] == True]

    if results.empty:
        return json.dumps({"message": "No car rentals found matching your criteria."})
    return results.head(5).to_json(orient="records", indent=2)

print("✅ Tool functions defined: search_flights, search_hotels, search_car_rentals")

## 4. Create the Agent with Tools

The MAF <code>Agent</code> class automatically handles tool registration and the tool-call loop: call model → detect tool calls → execute tools → return results → call model again. You pass in the Python functions.

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #1565c0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>💡 Key Concept:</b> Unlike prompted agents where Foundry orchestrates tool calls externally, MAF runs the entire tool-call loop in-process. This means lower latency and full access to local Python state.
</div>

---

In [ ]:
# Create the MAF Agent with local tool functions
from agent_framework import Agent
from agent_framework.azure import AzureAIAgentClient

# Create an Azure AI client for the Agent class
agent_client = AzureAIAgentClient(project_client=project_client)

# Create the agent with local tools — MAF handles the tool-call loop automatically
travel_agent = Agent(
    client=agent_client,
    name="contoso-travel-agent",
    model=model_name,
    instructions="""You are the Contoso Travel agent. Help customers find flights, hotels, and car rentals.
Use the search tools to look up real inventory data. Always provide specific options with prices.
Contoso Travel routes: Seattle↔Paris, NYC↔London, SF↔Tokyo, Chicago↔Rome, Denver↔Cancún.""",
    tools=[search_flights, search_hotels, search_car_rentals],
)

print("✅ Travel agent created with 3 local tools")

## 5. Test Flight Search

Send a single-tool query to verify the agent can call <code>search_flights</code> and return results.

---

In [ ]:
# Test a single-tool flight search query
from agent_framework import ChatMessage, Role, TextContent

# Test a flight search query
test_msg = ChatMessage(
    role=Role.USER,
    content=[TextContent(text="Find me flights from Seattle to Paris under $800")]
)

response = await travel_agent.run([test_msg])
print("✈️ Agent Response:")
print("-" * 50)
print(response.messages[-1].content[0].text)

## 6. Test Multi-Tool Query

Send a query that requires the agent to call multiple tools — flights, hotels, and car rentals — in a single turn.

---

In [ ]:
# Test a query that requires multiple tools
combo_msg = ChatMessage(
    role=Role.USER,
    content=[TextContent(text="I'm planning a trip to Tokyo. Find me flights from San Francisco, a 4-star hotel, and an SUV rental.")]
)

response = await travel_agent.run([combo_msg])
print("🧳 Multi-tool Response:")
print("-" * 50)
print(response.messages[-1].content[0].text)

## 7. Compare — Prompted vs MAF Tool Execution

| Aspect | Prompted (FunctionTool) | MAF (Local Tools) |
|--------|------------------------|-------------------|
| Execution | Foundry calls your function externally | Function runs in-process |
| Latency | Network round-trip per tool call | Local execution (faster) |
| Data access | Must expose data via function definitions | Direct access to DataFrames, databases, APIs |
| Complex logic | Limited to what JSON schema can express | Full Python — loops, ML models, custom algorithms |
| Schema generation | Manual JSON schema | Auto-generated from `Annotated` type hints |

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>💡 Tip:</b> MAF's <code>Annotated</code> type hints are a significant developer experience improvement — no manual JSON schema authoring. The framework introspects your function signatures and generates the tool definitions automatically.
</div>

---

## 8. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2e8b57; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>✅ What We Accomplished</b>
  <ul style="margin-bottom: 0;">
    <li>Defined <b>3 local tool functions</b> (<code>search_flights</code>, <code>search_hotels</code>, <code>search_car_rentals</code>) using <code>Annotated</code> type hints</li>
    <li>Created a MAF <code>Agent</code> with an <b>automatic tool-call loop</b> — no manual orchestration required</li>
    <li>Tested <b>single-tool</b> and <b>multi-tool</b> queries against Contoso Travel inventory</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #ff9800; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
  <b>➡️ Next:</b> In <a href="lab-04-tracing.ipynb">Lab 04</a>, we'll add <b>tracing</b> to the travel agent so you can see every tool call, token count, and latency metric in Azure Monitor.
</div>

In [ ]:
# Close client connections and release resources
openai_client.close()
project_client.close()
credential.close()
print("✅ Clients closed.")